In [1]:
import subprocess, sys, warnings
for p in ["laspy","lazrs"]:
    try: __import__(p)
    except: subprocess.run([sys.executable,"-m","pip","install",p,"-q"])
import numpy as np, laspy
from google.colab import files
from IPython.display import display, HTML
warnings.filterwarnings("ignore")
print("Ready. Run next cell to upload.")

Ready. Run next cell to upload.


In [4]:
print("Upload your .laz or .las:\n")
up = files.upload()
fname, content = next(iter(up.items()))
with open(fname,"wb") as f: f.write(content)

las = laspy.read(fname)
bm = las.classification == 6
xb=las.x[bm].copy(); yb=las.y[bm].copy(); zb=las.z[bm].copy()
rb=las.red[bm].astype(np.float32); th=rb/rb.max()

N=150000; n=len(xb)
if n>N: idx=np.random.RandomState(42).choice(n,N,replace=False)
else: idx=np.arange(n)
xb=xb[idx]-xb[idx].mean(); yb=yb[idx]-yb[idx].mean(); zb=zb[idx]-zb[idx].min(); th=th[idx]

pcts=np.percentile(th,[25,50,75])
zones=np.digitize(th,pcts)
zmeans=np.array([th[zones==z].mean() for z in range(4)])
zvars=np.zeros(len(th))
for z in range(4):
    m=zones==z; zvars[m]=th[m]-th[m].mean()

print(f"Points: {n:,} -> {len(idx):,}")
print("Ready for viewer.")

Upload your .laz or .las:



Saving 03232025_point_cloud (3).las to 03232025_point_cloud (3).las
Points: 2,571,091 -> 150,000
Ready for viewer.


In [5]:
# JS-powered 3D viewer with sliders — no ipywidgets, no Python callbacks
# Everything runs in browser via Plotly.js restyle() = instant recolor

import json
xj=json.dumps(xb.round(2).tolist())
yj=json.dumps(yb.round(2).tolist())
zj=json.dumps(zb.round(2).tolist())
zonesj=json.dumps(zones.tolist())
zvarsj=json.dumps(zvars.round(4).tolist())
zmeansj=json.dumps(zmeans.round(4).tolist())

html = f"""
<div id="vw" style="width:100%;height:620px"></div>
<div style="padding:12px 4px;font-family:sans-serif;font-size:13px;color:#aaa;display:flex;flex-wrap:wrap;gap:18px;align-items:center">
  <label style="display:flex;align-items:center;gap:6px">Temp (F)
    <input type="range" id="st" min="20" max="100" value="60" style="width:160px" oninput="this.nextElementSibling.textContent=this.value;rc()"><span>60</span></label>
  <label style="display:flex;align-items:center;gap:6px">Solar
    <input type="range" id="ss" min="0" max="1000" value="500" step="10" style="width:140px" oninput="this.nextElementSibling.textContent=this.value;rc()"><span>500</span></label>
  <label style="display:flex;align-items:center;gap:6px">Wind
    <input type="range" id="sw" min="0" max="25" value="5" style="width:120px" oninput="this.nextElementSibling.textContent=this.value;rc()"><span>5</span></label>
  <label style="display:flex;align-items:center;gap:6px">RH%
    <input type="range" id="sh" min="10" max="100" value="50" style="width:120px" oninput="this.nextElementSibling.textContent=this.value;rc()"><span>50</span></label>
</div>
<script src="https://cdn.jsdelivr.net/npm/plotly.js@2/dist/plotly.min.js"><\/script>
<script>
const x={xj},y={yj},z={zj},zn={zonesj},zv={zvarsj},zm={zmeansj};
const rT=58,rS=550,rW=5,rH=45;
const sn=[[.0008,.00005,-.002,-.0005],[.0015,.0001,-.003,.0003],[.002,.0002,-.004,-.0008],[.001,.00025,-.005,-.0004]];
function rc(){{const t=+document.getElementById('st').value,s=+document.getElementById('ss').value,
w=+document.getElementById('sw').value,h=+document.getElementById('sh').value;
const dt=t-rT,ds=s-rS,dw=w-rW,dh=h-rH;
const c=new Float32Array(zn.length);
for(let i=0;i<zn.length;i++){{const q=zn[i];c[i]=Math.max(0,Math.min(1,zm[q]+sn[q][0]*dt+sn[q][1]*ds+sn[q][2]*dw+sn[q][3]*dh+zv[i]*.4));}}
Plotly.restyle('vw',{{'marker.color':[Array.from(c)]}});}}
const ic=zn.map((q,i)=>Math.max(0,Math.min(1,zm[q]+zv[i]*.4)));
Plotly.newPlot('vw',[{{type:'scatter3d',mode:'markers',x,y,z,
marker:{{size:1.3,color:ic,colorscale:'Inferno',cmin:0,cmax:1,colorbar:{{title:'Thermal',thickness:12,len:.5}},opacity:.85}},
hovertemplate:'X:%{{x:.0f}} Y:%{{y:.0f}} Z:%{{z:.0f}}<extra><\/extra>'}}],
{{scene:{{xaxis:{{title:'X'}},yaxis:{{title:'Y'}},zaxis:{{title:'Z'}},aspectmode:'data',camera:{{eye:{{x:1.5,y:1.5,z:1}}}}}},
paper_bgcolor:'#1a1a2e',plot_bgcolor:'#1a1a2e',font:{{color:'#ccc'}},margin:{{l:0,r:0,t:10,b:0}},height:620}},
{{responsive:true}});
<\/script>"""

display(HTML(html))
print("Drag sliders. 3D recolors instantly (no Python round-trip).")
print("Rotate: drag | Zoom: scroll | Pan: right-drag")

Output hidden; open in https://colab.research.google.com to view.